# Thermal Analysis with Differential Scanning Calorimetry (DSC)

**Objective:** This lesson introduces Differential Scanning Calorimetry (DSC), a fundamental technique for measuring the thermal properties of materials, especially polymers. We will learn how to interpret a DSC thermogram to identify key thermal transitions like the glass transition, crystallization, and melting.

**Learning Goals:**
1.  Understand the principle of DSC: measuring the difference in heat flow between a sample and a reference.
2.  Learn to interpret a **DSC thermogram** (Heat Flow vs. Temperature).
3.  Identify the characteristic signatures of a **glass transition ($T_g$)**, an **exothermic crystallization peak ($T_c$)**, and an **endothermic melting peak ($T_m$)**.
4.  Use numerical derivatives (`np.gradient`) to help identify the midpoint of the glass transition.
5.  Use peak finding algorithms (`scipy.signal.find_peaks`) to precisely locate crystallization and melting temperatures.

## Part 1: The Theory - How DSC Works

A DSC works by taking a tiny sample of a material and a "reference" (usually an empty pan) and heating them at a precisely controlled rate (e.g., 10 °C/min). The instrument continuously measures the difference in heat flow required to keep the sample and the reference at the same temperature.

As the material undergoes a thermal transition, it will require either more or less heat than the reference. The resulting plot of Heat Flow vs. Temperature is called a **thermogram**.

*   **Glass Transition ($T_g$):** A change in the heat capacity of the amorphous parts of a polymer. Appears as a step-like change in the baseline.
*   **Crystallization ($T_c$):** An exothermic process where disordered polymer chains arrange into an ordered crystal. Appears as a peak pointing **down** (releasing heat).
*   **Melting ($T_m$):** An endothermic process where a crystalline structure breaks down. Appears as a peak pointing **up** (absorbing heat).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# --- Part 2: Simulate a DSC Thermogram ---
# We will simulate a thermogram for a semi-crystalline polymer like PET
# that is heated from below its glass transition temperature.

temperature = np.linspace(30, 300, 1000) # C

# 1. Create a baseline
baseline = 0.001 * temperature + 0.5

# 2. Add the glass transition (a step change modeled with a sigmoid)
Tg_midpoint = 80 # C
Tg_step_height = 0.3
glass_transition = Tg_step_height / (1 + np.exp(-(temperature - Tg_midpoint) / 3.0))

# 3. Add an exothermic crystallization peak (negative peak)
Tc = 130 # C
crystallization = -1.5 * np.exp(-((temperature - Tc) / 10)**2)

# 4. Add an endothermic melting peak (positive peak)
Tm = 255 # C
melting = 4.0 * np.exp(-((temperature - Tm) / 8)**2)

# Combine all features
heat_flow = baseline + glass_transition + crystallization + melting
heat_flow += np.random.normal(0, 0.03, size=heat_flow.shape) # Add noise

# --- Plot the Thermogram ---
plt.figure(figsize=(12, 6))
plt.plot(temperature, heat_flow)
plt.title('Simulated DSC Thermogram of a Polymer', fontsize=16, weight='bold')
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Heat Flow (W/g)', fontsize=12)
plt.text(85, 0.9, '$T_g$')
plt.text(135, 0, '$T_c$')
plt.text(260, 4.0, '$T_m$')
plt.grid(True)
plt.show()

## Part 3: Programmatic Analysis of Thermal Transitions

We can use numerical methods to automatically find these key transition points from the data.

In [ ]:
# --- 1. Find the Glass Transition (Tg) ---
# The glass transition is a step in the heat flow, which means it is a peak
# in the *first derivative* of the heat flow.
heat_flow_derivative = np.gradient(heat_flow, temperature)
tg_peak_index = np.argmax(heat_flow_derivative)
Tg = temperature[tg_peak_index]

# --- 2. Find the Crystallization (Tc) and Melting (Tm) Peaks ---
# Find exothermic peaks (negative) by finding peaks in the -1*heat_flow signal
tc_peak_indices, _ = find_peaks(-heat_flow, height=0.5)
Tc_found = temperature[tc_peak_indices][0]

# Find endothermic peaks (positive)
tm_peak_indices, _ = find_peaks(heat_flow, height=2.0)
Tm_found = temperature[tm_peak_indices][0]

print("--- Analysis Complete ---")
print(f"Glass Transition Temperature (Tg) found at: {Tg:.1f} °C")
print(f"Crystallization Temperature (Tc) found at: {Tc_found:.1f} °C")
print(f"Melting Temperature (Tm) found at:      {Tm_found:.1f} °C")

In [ ]:
# --- Visualize the Analysis ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot the main thermogram with found peaks
ax1.plot(temperature, heat_flow, label='Heat Flow')
ax1.plot(Tg, heat_flow[tg_peak_index], 'go', markersize=10, label=f'Tg = {Tg:.1f}°C')
ax1.plot(Tc_found, heat_flow[tc_peak_indices[0]], 'rs', markersize=10, label=f'Tc = {Tc_found:.1f}°C')
ax1.plot(Tm_found, heat_flow[tm_peak_indices[0]], 'b^', markersize=10, label=f'Tm = {Tm_found:.1f}°C')
ax1.set_ylabel('Heat Flow (W/g)')
ax1.set_title('DSC Thermogram with Identified Transitions')
ax1.legend()
ax1.grid(True)

# Plot the derivative to show how Tg was found
ax2.plot(temperature, heat_flow_derivative, label='Derivative of Heat Flow')
ax2.plot(Tg, heat_flow_derivative[tg_peak_index], 'go', markersize=10, label=f'Peak corresponds to Tg')
ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('d(Heat Flow)/dT')
ax2.set_title('Derivative Plot to Find Glass Transition')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Student Challenges

1.  **Degree of Crystallinity:** The area under the melting peak is the heat of fusion, $\Delta H_f$. This can be used to calculate the percent crystallinity of the polymer sample: 
$$ \% \text{Crystallinity} = \frac{\Delta H_f}{\Delta H_f^\circ} \times 100\% $$ 
where $\Delta H_f^\circ$ is the theoretical heat of fusion for a 100% crystalline sample (a known literature value). Use `scipy.integrate.simps` to calculate the area under the melting peak in our simulated data. If the $\Delta H_f^\circ$ for our polymer is 140 J/g, what is the crystallinity of our sample?

2.  **Heating Rate:** The appearance of a DSC thermogram is highly dependent on the heating/cooling rate. If you heat a sample very quickly, the molecules may not have time to rearrange and crystallize. How would the thermogram in Part 2 look if the heating rate was so fast that the crystallization peak ($T_c$) did not appear? Modify the code to remove that peak and observe the result.